# Speaker bootstrap, outlier sensitivity, feature-group analysis

These are original, unmodified Python source cells from the exported Kaggle notebook, organized by research phase. Execution outputs and counts have been cleared. Cells are in original index order within this phase. The Kaggle project workspace and some earlier phases are required; this notebook is not guaranteed to execute by itself from a clean environment. The original index for every cell is stored in cell metadata.

The historical record includes exploratory retries and some superseded checks. Do not rerun an early output-producing cell over frozen results without backing up the project first.

In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

SOURCES = {
    "A": ("baseline_acoustic_oof_predictions.csv",
          "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595",
          "acoustic_ridge_prediction"),
    "B1": ("b1_transcript_ridge_oof_predictions.csv",
           "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
           "b1_prediction"),
    "B2": ("b2_prompt_aware_ridge_oof_predictions.csv",
           "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
           "b2_prediction"),
    "C1": ("c1_fusion_ridge_oof_predictions.csv",
           "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
           "c1_prediction"),
    "C2": ("c2_prompt_aware_fusion_ridge_oof_predictions.csv",
           "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
           "c2_prediction")
}

AUDIT = P / "reports/step04_anomaly_classification_v1.csv"
AUDIT_SHA = "80122fe23528b9a221836ac1f738876a1ae503baa8dd3354cce329952e8952b0"

OUT = R / "step06_anomaly_sensitivity_400_399_396.csv"
PAIRED = R / "step06_anomaly_sensitivity_paired_differences.csv"
REPORT = P / "reports/step06_anomaly_sensitivity_v1.json"

def sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

assert sha256(AUDIT) == AUDIT_SHA

audit = read(AUDIT)

EXPECTED_CANDIDATES = {
    "030210164", "096340009",
    "096340012", "096340017"
}

# Verify the four previously declared anomaly candidates.
possible_flags = [
    c for c in audit.columns
    if "candidate" in c.lower()
    and ("sensitivity" in c.lower() or "anomaly" in c.lower())
]

if "sensitivity_analysis_candidate" in audit.columns:
    flag = "sensitivity_analysis_candidate"
elif len(possible_flags) == 1:
    flag = possible_flags[0]
else:
    raise ValueError(
        f"Review audit flag column: {audit.columns.tolist()}"
    )

values = audit[flag].astype(str).str.strip().str.lower()

candidate_ids = set(
    audit.loc[
        values.isin(["true", "1", "yes"]),
        "audio_id"
    ].astype(str)
)

assert candidate_ids == EXPECTED_CANDIDATES, (
    f"Audit candidates mismatch: {candidate_ids}"
)

KEYS = [
    "pilot_index", "audio_id", "speaker",
    "fluency", "validation_fold"
]
HKEYS = KEYS + ["hypothesis_id"]

frames = {}

for name, (filename, expected, col) in SOURCES.items():
    path = R / filename
    assert sha256(path) == expected, f"SHA mismatch: {name}"
    frames[name] = read(path)

base = frames["B1"][
    HKEYS + [SOURCES["B1"][2]]
].copy()

for name in ["B2", "C1", "C2"]:
    base = base.merge(
        frames[name][HKEYS + [SOURCES[name][2]]],
        on=HKEYS,
        validate="one_to_one"
    )

base = base.merge(
    frames["A"][KEYS + [SOURCES["A"][2]]],
    on=KEYS,
    validate="many_to_one"
)

assert len(base) == 2000
assert base["audio_id"].nunique() == 400
assert base.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

MODELS = list(SOURCES)
aggregations = {}

for name in MODELS:
    pred_col = SOURCES[name][2]

    base[f"{name}_ae"] = (
        base[pred_col] - base["fluency"]
    ).abs()

    base[f"{name}_se"] = (
        base[pred_col] - base["fluency"]
    ) ** 2

    aggregations[f"{name}_ae"] = (f"{name}_ae", "mean")
    aggregations[f"{name}_se"] = (f"{name}_se", "mean")
    aggregations[f"{name}_min"] = (pred_col, "min")
    aggregations[f"{name}_max"] = (pred_col, "max")

audio = base.groupby(
    KEYS, as_index=False
).agg(**aggregations)

assert len(audio) == 400
assert audio["audio_id"].is_unique

for name in MODELS:
    audio[f"{name}_range"] = (
        audio[f"{name}_max"] - audio[f"{name}_min"]
    )

assert np.allclose(audio["A_range"], 0, atol=1e-12)

CONDITIONS = {
    "primary_all_400": set(),
    "diagnostic_without_critical_399": {"030210164"},
    "diagnostic_without_4_candidates_396": candidate_ids
}

rows = []
pairs = []

PAIRINGS = [
    ("C2", "A"),
    ("C2", "C1"),
    ("B2", "B1")
]

for condition, excluded in CONDITIONS.items():
    sub = audio.loc[
        ~audio["audio_id"].isin(excluded)
    ].copy()

    assert len(sub) == 400 - len(excluded)

    for name in MODELS:
        rows.append({
            "condition": condition,
            "model": name,
            "audio_count": len(sub),
            "hypothesis_count": len(sub) * 5,
            "speaker_count": sub["speaker"].nunique(),
            "MAE": float(sub[f"{name}_ae"].mean()),
            "RMSE": float(
                np.sqrt(sub[f"{name}_se"].mean())
            ),
            "mean_prediction_range": float(
                sub[f"{name}_range"].mean()
            ),
            "median_prediction_range": float(
                sub[f"{name}_range"].median()
            ),
            "p95_prediction_range": float(
                sub[f"{name}_range"].quantile(0.95)
            )
        })

    for left, right in PAIRINGS:
        pairs.append({
            "condition": condition,
            "comparison": f"{left}_minus_{right}",
            "MAE_difference": float(
                (sub[f"{left}_ae"] -
                 sub[f"{right}_ae"]).mean()
            ),
            "mean_range_difference": float(
                (sub[f"{left}_range"] -
                 sub[f"{right}_range"]).mean()
            )
        })

result = pd.DataFrame(rows)
paired = pd.DataFrame(pairs)

assert len(result) == 15
assert len(paired) == 9

EXPECTED_PRIMARY = {
    "A": (0.9399, 1.2066),
    "B1": (1.2426, 1.9972),
    "B2": (1.1761, 1.9518),
    "C1": (0.8972, 1.3735),
    "C2": (0.7782, 1.2461)
}

for name, (mae, rmse) in EXPECTED_PRIMARY.items():
    row = result.loc[
        result["condition"].eq("primary_all_400") &
        result["model"].eq(name)
    ].iloc[0]

    assert np.isclose(row["MAE"], mae, atol=1e-4)
    assert np.isclose(row["RMSE"], rmse, atol=1e-4)

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(tmp, index=False, encoding="utf-8")

    if path.exists():
        assert sha256(path) == sha256(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_csv(result, OUT)
safe_csv(paired, PAIRED)

report = {
    "status": "PASS",
    "analysis": "preflagged_asr_anomaly_sensitivity",
    "primary_audio_count": 400,
    "primary_observations_retained": True,
    "refitting_performed": False,
    "candidate_audio_ids": sorted(candidate_ids),
    "audit_sha256": sha256(AUDIT),
    "conditions": {
        name: {
            "excluded_audio_ids": sorted(ids),
            "remaining_audio_count": 400 - len(ids)
        }
        for name, ids in CONDITIONS.items()
    },
    "main_results_sha256": sha256(OUT),
    "paired_results_sha256": sha256(PAIRED),
    "source_prediction_sha256": {
        name: sha256(R / spec[0])
        for name, spec in SOURCES.items()
    },
    "interpretation": (
        "Post-hoc sensitivity analysis using previously "
        "flagged ASR anomalies. No retraining and no "
        "primary-analysis exclusions. Reading prompt "
        "disagreement is not verified WER or CER."
    )
}

if REPORT.exists():
    old = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert old == report, (
        "Existing sensitivity report differs."
    )
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step6_anomaly_sensitivity"] = "qc_pass"
progress["step6_anomaly_sensitivity_sha256"] = sha256(OUT)
progress["next_task"] = (
    "Prepare integrated final model, uncertainty and "
    "robustness result tables"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.2 ANOMALY SENSITIVITY AUDIT ===")
print("Status: PASS")
print("Pre-flagged candidates:", sorted(candidate_ids))
print("Primary observations removed: NONE")
print("Model refitting: NO")

print("\n=== PRIMARY VS SENSITIVITY METRICS ===")
print(
    result[
        [
            "condition", "model", "audio_count",
            "MAE", "RMSE", "mean_prediction_range"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== PAIRED MODEL DIFFERENCES ===")
print(paired.round(4).to_string(index=False))

print("\nResults:", OUT)
print("SHA256:", sha256(OUT))
print("Paired results:", PAIRED)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
FINAL = P / "results/step06_final_tables"
CHECK = P / "checkpoints/research_progress.json"

BOOT = R / "speaker_cluster_bootstrap_10000_ci.csv"
SENS = R / "step06_anomaly_sensitivity_400_399_396.csv"
PAIRED = R / "step06_anomaly_sensitivity_paired_differences.csv"
SENS_REPORT = P / "reports/step06_anomaly_sensitivity_v1.json"

EXPECTED_BOOT = (
    "5ef3d6dd01eb05758ccb8b4f77293396a81e988b9b9e33152cb666c6b48d1d93"
)
EXPECTED_SENS = (
    "83b34da4aa34634319b59c94391967565e3f3c0fd438fb5a80d48131ca5563d3"
)

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert sha(BOOT) == EXPECTED_BOOT
assert sha(SENS) == EXPECTED_SENS

sens_report = json.loads(
    SENS_REPORT.read_text(encoding="utf-8")
)
assert sens_report["status"] == "PASS"
assert sens_report["main_results_sha256"] == sha(SENS)
assert sens_report["paired_results_sha256"] == sha(PAIRED)

boot = pd.read_csv(BOOT)
sens = pd.read_csv(SENS)
paired = pd.read_csv(PAIRED)

MODELS = ["A", "B1", "B2", "C1", "C2"]
METRICS = ["MAE", "RMSE", "mean_range"]

assert len(boot) == 33
assert len(sens) == 15
assert len(paired) == 9

# 1. Primary model metrics with cluster-bootstrap CIs.
primary = sens.loc[
    sens["condition"] == "primary_all_400"
].set_index("model")

assert set(primary.index) == set(MODELS)

def get_boot(name, metric):
    row = boot.loc[
        boot["comparison"].eq(name)
        & boot["metric"].eq(metric)
    ]
    assert len(row) == 1
    return row.iloc[0]

main_rows = []

feature_counts = {
    "A": 14,
    "B1": 7,
    "B2": 9,
    "C1": 21,
    "C2": 23
}

for model in MODELS:
    row = primary.loc[model]

    mae = get_boot(model, "MAE")
    rmse = get_boot(model, "RMSE")
    stability = get_boot(model, "mean_range")

    assert np.isclose(
        float(row["MAE"]), float(mae["observed"]),
        atol=1e-10
    )
    assert np.isclose(
        float(row["RMSE"]), float(rmse["observed"]),
        atol=1e-10
    )
    assert np.isclose(
        float(row["mean_prediction_range"]),
        float(stability["observed"]),
        atol=1e-10
    )

    main_rows.append({
        "model": model,
        "feature_count": feature_counts[model],
        "MAE": float(mae["observed"]),
        "MAE_CI_low": float(mae["ci_2_5"]),
        "MAE_CI_high": float(mae["ci_97_5"]),
        "RMSE": float(rmse["observed"]),
        "RMSE_CI_low": float(rmse["ci_2_5"]),
        "RMSE_CI_high": float(rmse["ci_97_5"]),
        "mean_prediction_range": float(
            stability["observed"]
        ),
        "range_CI_low": float(stability["ci_2_5"]),
        "range_CI_high": float(stability["ci_97_5"])
    })

main_table = pd.DataFrame(main_rows)

# 2. Paired differences and bootstrap CIs.
COMPARISONS = [
    "C1_minus_A",
    "C2_minus_A",
    "C2_minus_C1",
    "B2_minus_B1",
    "C1_minus_B1",
    "C2_minus_B2"
]

comparison_rows = []

for name in COMPARISONS:
    for metric in METRICS:
        row = get_boot(name, metric)

        comparison_rows.append({
            "comparison": name,
            "metric": metric,
            "difference": float(row["observed"]),
            "CI_low": float(row["ci_2_5"]),
            "CI_high": float(row["ci_97_5"]),
            "CI_excludes_zero": bool(
                float(row["ci_2_5"]) > 0
                or float(row["ci_97_5"]) < 0
            )
        })

comparison_table = pd.DataFrame(comparison_rows)
assert len(comparison_table) == 18

# 3. Sensitivity matrix, unchanged from source.
condition_order = [
    "primary_all_400",
    "diagnostic_without_critical_399",
    "diagnostic_without_4_candidates_396"
]

sensitivity_table = sens.copy()
sensitivity_table["condition"] = pd.Categorical(
    sensitivity_table["condition"],
    categories=condition_order,
    ordered=True
)
sensitivity_table["model"] = pd.Categorical(
    sensitivity_table["model"],
    categories=MODELS,
    ordered=True
)
sensitivity_table = sensitivity_table.sort_values(
    ["condition", "model"]
).reset_index(drop=True)

assert len(sensitivity_table) == 15

# 4. Save deterministic, immutable results.
FINAL.mkdir(parents=True, exist_ok=True)

OUTPUTS = {
    "table01_primary_models_with_ci.csv": main_table,
    "table02_paired_differences_with_ci.csv": comparison_table,
    "table03_anomaly_sensitivity.csv": sensitivity_table,
    "table04_sensitivity_paired_differences.csv": paired
}

def safe_save(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp,
        index=False,
        encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

for filename, frame in OUTPUTS.items():
    safe_save(frame, FINAL / filename)

manifest = {
    "status": "PASS",
    "analysis": "integrated_fixed_alpha_ridge_results",
    "unique_audios": 400,
    "unique_speakers": 20,
    "asr_hypotheses": 2000,
    "models": MODELS,
    "speaker_disjoint_cv": "frozen_5fold",
    "bootstrap_clusters": 20,
    "bootstrap_iterations": 10000,
    "confidence_interval_method": (
        "paired_speaker_cluster_percentile_bootstrap"
    ),
    "all_primary_observations_retained": True,
    "model_refitting": False,
    "hyperparameter_tuning": False,
    "ridge_alpha": 1.0,
    "original_source_sha256": {
        "bootstrap": sha(BOOT),
        "sensitivity": sha(SENS),
        "sensitivity_paired": sha(PAIRED),
        "sensitivity_report": sha(SENS_REPORT)
    },
    "generated_tables_sha256": {
        filename: sha(FINAL / filename)
        for filename in OUTPUTS
    },
    "limitations": [
        "Pilot sample contains only 20 speakers.",
        "Bootstrap intervals are conditional on saved OOF predictions.",
        "ASR hypotheses from one audio are correlated.",
        "Prompt-ASR disagreement is not verified WER/CER.",
        "Reading prompts are not verified verbatim transcripts.",
        "Sensitivity exclusions are diagnostic, not primary.",
        "Official test split remains untouched.",
        "Results use fixed alpha=1.0 without hyperparameter tuning."
    ]
}

MANIFEST = FINAL / "step06_integrated_results_manifest.json"

if MANIFEST.exists():
    old = json.loads(
        MANIFEST.read_text(encoding="utf-8")
    )
    assert old == manifest, (
        "Existing manifest differs; preserved."
    )
else:
    tmp = MANIFEST.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(manifest, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(MANIFEST)

progress = json.loads(
    CHECK.read_text(encoding="utf-8")
)
progress["step6_integrated_results"] = "qc_pass"
progress["step6_integrated_manifest_sha256"] = sha(MANIFEST)
progress["next_task"] = (
    "Review consolidated results, evaluate remaining "
    "methodological limitations and prepare research reporting"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()
CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.3 INTEGRATED RESULTS ===")
print("Status: PASS")
print("Models:", len(MODELS))
print("Primary audios:", 400)
print("Speakers:", 20)
print("Primary observations excluded: NONE")

print("\n=== TABLE 1: PRIMARY MODELS WITH 95% CI ===")
print(main_table.round(4).to_string(index=False))

print("\n=== TABLE 2: PAIRED DIFFERENCES WITH 95% CI ===")
print(
    comparison_table.round(4).to_string(index=False)
)

print("\n=== TABLE 3: ANOMALY SENSITIVITY ===")
print(
    sensitivity_table[
        [
            "condition", "model", "audio_count",
            "MAE", "RMSE", "mean_prediction_range"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== SAVED RESEARCH TABLES ===")
for filename in OUTPUTS:
    path = FINAL / filename
    print(filename, "| SHA256:", sha(path))

print("\nManifest:", MANIFEST)
print("Manifest SHA256:", sha(MANIFEST))
print("Checkpoint updated.")


In [ ]:

import hashlib
import json
import re
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
R = P / "results/step05"
CHECK = P / "checkpoints/research_progress.json"

ASR = P / "features/asr/full/asr_hypotheses_0001-0400.csv"

FILES = {
    "B1": R / "b1_transcript_ridge_oof_predictions.csv",
    "B2": R / "b2_prompt_aware_ridge_oof_predictions.csv",
    "C1": R / "c1_fusion_ridge_oof_predictions.csv",
    "C2": R / "c2_prompt_aware_fusion_ridge_oof_predictions.csv"
}

EXPECTED = {
    ASR: "b67dfe5cca2329407ab07658809434c2e3bad00d227485f3f8c8c9c1dbbf3e80",
    FILES["B1"]: "375ac49b0d5b565adf65d04407d6afc1816f64f5ba108afef8d22458383807dc",
    FILES["B2"]: "ba212868934dbe4d92234f46ccb5f7d7e66a56c781d74aeb24b310b722cfb189",
    FILES["C1"]: "43aa2b8d73c60e44ee4ee2f996f5b243ae2a09c5b9a78f909b2a56ba7306669e",
    FILES["C2"]: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

OUT = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
SUMMARY = P / "results/step06_final_tables/step06_variability_association.csv"
REPORT = P / "reports/step06_asr_variability_association_v1.json"

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file(), f"Missing: {path}"
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"},
        keep_default_na=False
    )

asr = read(ASR)
assert len(asr) == 2000

TOKEN_PATTERN = re.compile(
    r"[^\W_]+(?:['’][^\W_]+)*",
    flags=re.UNICODE
)

def lexical_normalize(text):
    return " ".join(
        w.casefold() for w in TOKEN_PATTERN.findall(str(text))
    )

asr["lexical_text"] = asr["transcription"].map(lexical_normalize)
asr["raw_text"] = asr["transcription"].astype(str).str.strip()

ID = ["pilot_index", "audio_id", "speaker"]
KEYS = ID + ["hypothesis_id"]

assert not asr.duplicated(KEYS).any()
assert asr.groupby("audio_id")["hypothesis_id"].nunique().eq(5).all()

audio = asr.groupby(ID, as_index=False).agg(
    raw_unique=("raw_text", "nunique"),
    lexical_unique=("lexical_text", "nunique"),
    hypothesis_count=("hypothesis_id", "count")
)

assert len(audio) == 400
assert audio["hypothesis_count"].eq(5).all()

audio["raw_variable"] = audio["raw_unique"] > 1
audio["lexical_variable"] = audio["lexical_unique"] > 1

PRED_COLS = {
    "B1": "b1_prediction",
    "B2": "b2_prediction",
    "C1": "c1_prediction",
    "C2": "c2_prediction"
}

for model, path in FILES.items():
    df = read(path)
    col = PRED_COLS[model]
    df[col] = pd.to_numeric(df[col], errors="raise")

    assert len(df) == 2000
    assert not df.duplicated(KEYS).any()

    stat = df.groupby(ID, as_index=False).agg(
        pred_min=(col, "min"),
        pred_max=(col, "max"),
        pred_sd=(col, "std"),
        rows=(col, "count")
    )
    assert len(stat) == 400
    assert stat["rows"].eq(5).all()

    stat[f"{model}_range"] = stat["pred_max"] - stat["pred_min"]
    stat[f"{model}_sd"] = stat["pred_sd"]

    audio = audio.merge(
        stat[ID + [f"{model}_range", f"{model}_sd"]],
        on=ID,
        validate="one_to_one"
    )

assert len(audio) == 400
assert audio["speaker"].nunique() == 20

# Independently reproduce previously reported counts.
assert int(audio["raw_variable"].sum()) == 224
assert int(audio["lexical_variable"].sum()) == 172

observations = []

for model in PRED_COLS:
    col = f"{model}_range"

    rho, naive_p = spearmanr(
        audio["lexical_unique"].to_numpy(),
        audio[col].to_numpy()
    )

    group_means = audio.groupby(
        "lexical_variable"
    )[col].mean()

    observations.append({
        "model": model,
        "n_audios": 400,
        "spearman_rho_lexical_unique_vs_range": float(rho),
        "naive_p_not_for_inference": float(naive_p),
        "mean_range_lexically_invariant": float(
            group_means.loc[False]
        ),
        "mean_range_lexically_variable": float(
            group_means.loc[True]
        ),
        "mean_range_all": float(audio[col].mean()),
        "nonzero_range_audios": int(
            (audio[col] > 1e-10).sum()
        )
    })

summary = pd.DataFrame(observations)

# Invariant normalized transcripts should have exactly
# invariant linguistic features and model predictions.
# Verify this structural expectation; any violation
# requires investigation.
invariant = audio.loc[~audio["lexical_variable"]]

for model in PRED_COLS:
    assert np.allclose(
        invariant[f"{model}_range"],
        0,
        atol=1e-9
    ), f"Unexpected variability in {model} for invariant text"

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(temp, index=False, encoding="utf-8")
    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing file differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

safe_csv(audio.sort_values("pilot_index"), OUT)
safe_csv(summary, SUMMARY)

report = {
    "status": "PASS",
    "analysis": "asr_lexical_variability_vs_prediction_range",
    "audio_count": 400,
    "speaker_count": 20,
    "raw_variable_audios": int(audio["raw_variable"].sum()),
    "lexically_variable_audios": int(
        audio["lexical_variable"].sum()
    ),
    "correlation_type": "Spearman descriptive",
    "naive_p_values_for_inference": False,
    "speaker_clustered_uncertainty_estimated": False,
    "models_retrained": False,
    "all_original_observations_retained": True,
    "source_sha256": {
        str(path.relative_to(P)): sha(path)
        for path in EXPECTED
    },
    "audio_csv_sha256": sha(OUT),
    "summary_csv_sha256": sha(SUMMARY),
    "interpretation": (
        "Observed association, not causal evidence. "
        "Zero prediction range for lexically invariant "
        "hypotheses is expected from deterministic features. "
        "Speaker-clustered uncertainty is still required."
    )
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_asr_variability_association"] = "qc_pass"
progress["step6_asr_variability_association_sha256"] = sha(SUMMARY)
progress["next_task"] = (
    "Review ASR variability associations and assess "
    "speaker-clustered uncertainty and interpretation"
)
progress["updated_at"] = datetime.now(timezone.utc).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.4 ASR VARIABILITY ASSOCIATION ===")
print("Status: PASS")
print("Audios:", len(audio))
print("Speakers:", audio["speaker"].nunique())
print("Raw-variable audios:", int(audio["raw_variable"].sum()))
print("Lexically-variable audios:", int(audio["lexical_variable"].sum()))
print("Model refitting: NO")

print("\n=== VARIABILITY VS PREDICTION RANGE ===")
print(summary.round(4).to_string(index=False))

print("\n=== LEXICAL UNIQUENESS DISTRIBUTION ===")
print(
    audio["lexical_unique"]
    .value_counts()
    .sort_index()
    .to_string()
)

print("\n=== TOP 10 C2 PREDICTION RANGES ===")
print(
    audio.nlargest(10, "C2_range")[
        ["audio_id", "speaker", "lexical_unique",
         "B1_range", "C1_range", "C2_range"]
    ].round(4).to_string(index=False)
)

print("\nAudio CSV:", OUT)
print("Summary CSV:", SUMMARY)
print("Report:", REPORT)
print("Checkpoint updated.")



In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from scipy.stats import spearmanr

P = Path("/kaggle/working/acoustic_linguistic_research")
SOURCE = P / "results/step06_final_tables/step06_asr_variability_per_audio.csv"
CHECK = P / "checkpoints/research_progress.json"

OUT = P / "results/step06_final_tables/step06_variable_subset_bootstrap.csv"
REPORT = P / "reports/step06_variable_subset_bootstrap_v1.json"

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

assert SOURCE.is_file()
df = pd.read_csv(
    SOURCE,
    dtype={"audio_id": "string", "speaker": "string"}
)

assert len(df) == 400
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

MODELS = ["B1", "B2", "C1", "C2"]
for model in MODELS:
    assert f"{model}_range" in df.columns

assert "lexical_unique" in df.columns

# The main analysis is restricted to genuinely
# variable normalized ASR hypotheses.
variable = df.loc[df["lexical_unique"] > 1].copy()

assert len(variable) == 172
assert variable["audio_id"].is_unique

speaker_ids = sorted(df["speaker"].astype(str).unique())
assert len(speaker_ids) == 20

variable_speakers = sorted(variable["speaker"].astype(str).unique())

# Speaker bootstrap of the complete observed variable
# subset, keeping all selected audios of a speaker together.
groups = {
    s: variable.loc[
        variable["speaker"] == s
    ].copy()
    for s in speaker_ids
}

N_BOOT = 10000
SEED = 20261026
rng = np.random.default_rng(SEED)

draws = rng.integers(0, len(speaker_ids), size=(N_BOOT, 20))

ANOMALIES = {
    "030210164",
    "096340009",
    "096340012",
    "096340017"
}

def rho_or_nan(frame, model):
    x = frame["lexical_unique"].to_numpy(dtype=float)
    y = frame[f"{model}_range"].to_numpy(dtype=float)

    if len(frame) < 5:
        return float("nan")

    if np.unique(x).size < 2 or np.unique(y).size < 2:
        return float("nan")

    return float(spearmanr(x, y).statistic)

rows = []

for model in MODELS:

    observed = rho_or_nan(variable, model)
    assert np.isfinite(observed)

    samples = []

    for draw in draws:
        pieces = [
            groups[speaker_ids[int(i)]]
            for i in draw
            if len(groups[speaker_ids[int(i)]]) > 0
        ]

        if not pieces:
            continue

        sample = pd.concat(pieces, ignore_index=True)
        r = rho_or_nan(sample, model)

        if np.isfinite(r):
            samples.append(r)

    samples = np.asarray(samples, dtype=float)
    assert len(samples) >= 9500, (
        f"Too few valid bootstrap draws: {model}"
    )

    filtered = variable.loc[
        ~variable["audio_id"].isin(ANOMALIES)
    ]

    diagnostic_rho = rho_or_nan(filtered, model)

    rows.append({
        "model": model,
        "subset": "lexically_variable_172",
        "unique_audios": len(variable),
        "speaker_clusters_present": len(variable_speakers),
        "observed_spearman_rho": observed,
        "bootstrap_ci_low": float(
            np.quantile(samples, 0.025)
        ),
        "bootstrap_ci_high": float(
            np.quantile(samples, 0.975)
        ),
        "valid_bootstrap_draws": len(samples),
        "mean_prediction_range": float(
            variable[f"{model}_range"].mean()
        ),
        "diagnostic_n_without_4": len(filtered),
        "diagnostic_rho_without_4": diagnostic_rho
    })

results = pd.DataFrame(rows)

assert len(results) == 4
assert results["observed_spearman_rho"].notna().all()

OUT.parent.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

temp = OUT.with_suffix(".tmp")
results.to_csv(
    temp,
    index=False,
    encoding="utf-8",
    float_format="%.12g"
)

if OUT.exists():
    assert sha(OUT) == sha(temp), (
        "Existing results differ; original preserved."
    )
    temp.unlink()
else:
    temp.replace(OUT)

report = {
    "status": "PASS",
    "analysis": "lexically_variable_subset_speaker_bootstrap",
    "source_sha256": sha(SOURCE),
    "audio_count_full": 400,
    "audio_count_variable": 172,
    "speaker_clusters_total": 20,
    "speaker_clusters_with_variable_audio": len(variable_speakers),
    "bootstrap_iterations": N_BOOT,
    "random_seed": SEED,
    "confidence_interval_method": (
        "speaker_cluster_percentile_bootstrap"
    ),
    "bootstrap_model_refitting": False,
    "excluded_from_primary_model_evaluation": False,
    "diagnostic_anomaly_audio_ids": sorted(ANOMALIES),
    "output_sha256": sha(OUT),
    "limitations": [
        "Analysis is conditional on having lexically variable ASR outputs.",
        "Spearman association does not establish causality.",
        "The same ASR outputs construct both lexical features and prediction variation.",
        "Bootstrap resamples speakers, not independent hypothesis rows.",
        "Only 20 speakers are available in the full pilot.",
        "No fitted model is retrained during bootstrap.",
        "Four-anomaly exclusion is sensitivity analysis only."
    ]
}

if REPORT.exists():
    old = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert old == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_variable_subset_bootstrap"] = "qc_pass"
progress["step6_variable_subset_bootstrap_sha256"] = sha(OUT)
progress["next_task"] = (
    "Interpret within-variable-audio ASR association, "
    "then assess feature contributions and reporting limitations"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.5 VARIABLE-SUBSET BOOTSTRAP ===")
print("Status: PASS")
print("Full audios: 400")
print("Lexically-variable audios:", len(variable))
print("Speakers represented:", len(variable_speakers))
print("Bootstrap iterations:", N_BOOT)
print("Model refitting: NO")

print("\n=== WITHIN-VARIABLE CORRELATION AND 95% CI ===")
print(
    results[
        [
            "model", "unique_audios",
            "observed_spearman_rho",
            "bootstrap_ci_low",
            "bootstrap_ci_high",
            "valid_bootstrap_draws"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== SENSITIVITY WITHOUT FOUR ANOMALY CANDIDATES ===")
print(
    results[
        [
            "model", "diagnostic_n_without_4",
            "observed_spearman_rho",
            "diagnostic_rho_without_4"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== VARIABLE-SUBSET MEAN PREDICTION RANGE ===")
print(
    results[
        ["model", "mean_prediction_range"]
    ].round(4).to_string(index=False)
)

print("\nOutput:", OUT)
print("SHA256:", sha(OUT))
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
SAVED = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
COEFS = OUTDIR / "step06_c2_standardized_coefficients.csv"
CONTRIB = OUTDIR / "step06_c2_critical_h2_vs_h1_contributions.csv"
REPORT = P / "reports/step06_c2_feature_contribution_audit.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    SAVED: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9"
}

def sha(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
saved = read(SAVED)

design = json.loads(DESIGN.read_text(encoding="utf-8"))

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]

FEATURES = ACOUSTIC + LING + PROMPT

assert len(ACOUSTIC) == 14
assert len(LING) == 7
assert len(PROMPT) == 2
assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one"
)

assert len(df) == 2000
assert len(saved) == 2000

key_full = KEYS + ["fluency", "validation_fold"]
assert not saved.duplicated(key_full).any()

coef_rows = []
pred_parts = []
critical_contributions = []

CRITICAL_AUDIO = "030210164"

for fold in range(1, 6):

    train = df[df["validation_fold"] != fold]
    val = df[df["validation_fold"] == fold]

    assert len(train) == 1600
    assert len(val) == 400
    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("ridge", Ridge(alpha=1.0))
    ])

    model.fit(train[FEATURES], train["fluency"])

    predictions = model.predict(val[FEATURES])

    part = val[key_full].copy()
    part["reconstructed_prediction"] = predictions
    pred_parts.append(part)

    ridge = model.named_steps["ridge"]
    coefs = ridge.coef_

    for feature, coef in zip(FEATURES, coefs):
        if feature in ACOUSTIC:
            group = "acoustic"
        elif feature in LING:
            group = "linguistic"
        else:
            group = "prompt"

        coef_rows.append({
            "fold": fold,
            "feature": feature,
            "feature_group": group,
            "standardized_coefficient": float(coef),
            "absolute_coefficient": float(abs(coef))
        })

    if CRITICAL_AUDIO not in set(val["audio_id"]):
        continue

    critical = val.loc[
        val["audio_id"] == CRITICAL_AUDIO
    ].sort_values("hypothesis_id")

    assert len(critical) == 5
    assert critical["hypothesis_id"].tolist() == [
        "H1", "H2", "H3", "H4", "H5"
    ]

    # Same fitted fold model for H1 and H2.
    X1 = critical.loc[
        critical["hypothesis_id"] == "H1", FEATURES
    ]
    X2 = critical.loc[
        critical["hypothesis_id"] == "H2", FEATURES
    ]

    imputer = model.named_steps["imputer"]
    scaler = model.named_steps["scaler"]

    z1 = scaler.transform(imputer.transform(X1))[0]
    z2 = scaler.transform(imputer.transform(X2))[0]

    pred1 = float(model.predict(X1)[0])
    pred2 = float(model.predict(X2)[0])

    delta = (z2 - z1) * coefs

    assert np.isclose(
        delta.sum(), pred2 - pred1, atol=1e-9
    ), "Coefficient decomposition mismatch"

    for j, feature in enumerate(FEATURES):
        group = (
            "acoustic" if feature in ACOUSTIC else
            "linguistic" if feature in LING else "prompt"
        )

        critical_contributions.append({
            "fold": fold,
            "audio_id": CRITICAL_AUDIO,
            "comparison": "H2_minus_H1",
            "feature": feature,
            "feature_group": group,
            "h1_standardized_value": float(z1[j]),
            "h2_standardized_value": float(z2[j]),
            "standardized_coefficient": float(coefs[j]),
            "prediction_delta_contribution": float(delta[j]),
            "absolute_delta_contribution": float(abs(delta[j]))
        })

reconstructed = pd.concat(pred_parts, ignore_index=True)

verified = reconstructed.merge(
    saved[key_full + ["c2_prediction"]],
    on=key_full,
    how="inner",
    validate="one_to_one"
)

assert len(verified) == 2000

differences = np.abs(
    verified["reconstructed_prediction"]
    - verified["c2_prediction"]
)

MAX_DIFFERENCE = float(differences.max())

assert MAX_DIFFERENCE < 1e-7, (
    f"Reconstruction mismatch: {MAX_DIFFERENCE}"
)

coef_df = pd.DataFrame(coef_rows)
contrib_df = pd.DataFrame(critical_contributions)

assert len(coef_df) == 5 * 23
assert len(contrib_df) == 23
assert contrib_df["fold"].nunique() == 1
assert int(contrib_df["fold"].iloc[0]) == 2

critical_predictions = verified.loc[
    verified["audio_id"] == CRITICAL_AUDIO
].set_index("hypothesis_id")

critical_h1 = float(
    critical_predictions.loc["H1", "c2_prediction"]
)
critical_h2 = float(
    critical_predictions.loc["H2", "c2_prediction"]
)

assert np.isclose(
    critical_h2, -26.126716638013125, atol=1e-7
)

delta_total = float(
    contrib_df["prediction_delta_contribution"].sum()
)

assert np.isclose(
    critical_h2 - critical_h1,
    delta_total, atol=1e-7
)

group_contributions = (
    contrib_df.groupby("feature_group")[
        "prediction_delta_contribution"
    ].sum()
)

# Same audio means acoustic features cannot explain
# the difference between H1 and H2 predictions.
assert np.isclose(
    group_contributions["acoustic"],
    0,
    atol=1e-9
)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    temp = path.with_suffix(".tmp")
    frame.to_csv(
        temp, index=False, encoding="utf-8",
        float_format="%.12g"
    )

    if path.exists():
        assert sha(path) == sha(temp), (
            f"Existing output differs: {path}"
        )
        temp.unlink()
    else:
        temp.replace(path)

save_csv(coef_df, COEFS)
save_csv(contrib_df, CONTRIB)

report = {
    "status": "PASS",
    "analysis": "C2_ridge_feature_contribution_and_reconstruction",
    "models_retrained_for_reconstruction": True,
    "hyperparameters_changed": False,
    "speaker_disjoint_folds": 5,
    "OOF_predictions_verified": 2000,
    "maximum_reconstruction_difference": MAX_DIFFERENCE,
    "feature_count": 23,
    "critical_audio": CRITICAL_AUDIO,
    "critical_fold": 2,
    "critical_H1_prediction": critical_h1,
    "critical_H2_prediction": critical_h2,
    "H2_minus_H1_prediction_delta": delta_total,
    "group_contributions": {
        name: float(value)
        for name, value in group_contributions.items()
    },
    "coefficients_sha256": sha(COEFS),
    "contributions_sha256": sha(CONTRIB),
    "original_prediction_sha256": sha(SAVED),
    "interpretation": (
        "Exact linear decomposition of one H2-minus-H1 "
        "prediction difference under the same fitted Ridge "
        "model. Coefficient contributions are model-specific, "
        "not causal feature effects. Correlated features "
        "limit independent feature importance interpretation."
    )
}

if REPORT.exists():
    previous = json.loads(
        REPORT.read_text(encoding="utf-8")
    )
    assert previous == report, (
        "Existing report differs; original preserved."
    )
else:
    temp = REPORT.with_suffix(".tmp")
    temp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    temp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_c2_feature_contribution"] = "qc_pass"
progress["step6_c2_feature_contribution_sha256"] = sha(CONTRIB)
progress["next_task"] = (
    "Review critical feature contributions and "
    "coefficient consistency across speaker folds"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.6 C2 FEATURE CONTRIBUTION AUDIT ===")
print("Status: PASS")
print("Reconstructed OOF predictions:", len(verified))
print("Max prediction difference:", MAX_DIFFERENCE)
print("Critical audio:", CRITICAL_AUDIO)
print("Critical fold: 2")

print("\n=== CRITICAL H1 VS H2 ===")
print("H1 prediction:", round(critical_h1, 4))
print("H2 prediction:", round(critical_h2, 4))
print("H2 - H1 delta:", round(delta_total, 4))

print("\n=== CONTRIBUTION BY FEATURE GROUP ===")
print(group_contributions.round(4).to_string())

print("\n=== TOP 12 FEATURES EXPLAINING H2-H1 DELTA ===")
print(
    contrib_df.nlargest(
        12, "absolute_delta_contribution"
    )[
        [
            "feature", "feature_group",
            "h1_standardized_value",
            "h2_standardized_value",
            "standardized_coefficient",
            "prediction_delta_contribution"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== C2 TOP COEFFICIENTS ACROSS FIVE FOLDS ===")
coef_summary = coef_df.groupby(
    ["feature", "feature_group"]
).agg(
    mean_coefficient=("standardized_coefficient", "mean"),
    sd_coefficient=("standardized_coefficient", "std"),
    mean_absolute_coefficient=("absolute_coefficient", "mean")
).reset_index()

print(
    coef_summary.nlargest(
        12, "mean_absolute_coefficient"
    ).round(4).to_string(index=False)
)

print("\nCoefficients CSV:", COEFS)
print("Contributions CSV:", CONTRIB)
print("Report:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
COEFS = P / "results/step06_final_tables/step06_c2_standardized_coefficients.csv"
CONTRIB = P / "results/step06_final_tables/step06_c2_critical_h2_vs_h1_contributions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
STABILITY = OUTDIR / "step06_c2_coefficient_stability.csv"
CORRELATIONS = OUTDIR / "step06_c2_fold2_feature_correlations.csv"
EXTREMES = OUTDIR / "step06_c2_critical_h2_feature_extremes.csv"
REPORT = P / "reports/step06_c2_coefficient_stability_audit.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

assert COEFS.is_file()
assert CONTRIB.is_file()

coefs = pd.read_csv(COEFS)
contrib = pd.read_csv(CONTRIB)

assert len(coefs) == 115
assert len(contrib) == 23
assert coefs["fold"].nunique() == 5
assert coefs.groupby("feature")["fold"].nunique().eq(5).all()

# Analyze coefficient consistency across folds.
rows = []

for (feature, group), part in coefs.groupby(
    ["feature", "feature_group"]
):
    values = part.sort_values("fold")[
        "standardized_coefficient"
    ].to_numpy(dtype=float)

    nonzero = values[np.abs(values) > 1e-10]

    positive = int((values > 1e-10).sum())
    negative = int((values < -1e-10).sum())

    rows.append({
        "feature": feature,
        "feature_group": group,
        "mean_coefficient": float(values.mean()),
        "sd_coefficient": float(values.std(ddof=1)),
        "min_coefficient": float(values.min()),
        "max_coefficient": float(values.max()),
        "mean_absolute_coefficient": float(
            np.abs(values).mean()
        ),
        "positive_folds": positive,
        "negative_folds": negative,
        "sign_consistent": bool(
            len(nonzero) == 5 and
            (positive == 5 or negative == 5)
        )
    })

coef_stability = pd.DataFrame(rows)
assert len(coef_stability) == 23

data = pd.read_csv(
    DATA,
    dtype={"audio_id": "string", "speaker": "string"}
)
split = pd.read_csv(
    SPLIT,
    dtype={"audio_id": "string", "speaker": "string"}
)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["validation_fold"]],
    on=IDS,
    validate="many_to_one"
)

assert len(df) == 2000

# Use fold 2 training data only.
train = df.loc[df["validation_fold"] != 2]
val = df.loc[df["validation_fold"] == 2]

assert len(train) == 1600
assert len(val) == 400
assert set(train["speaker"]).isdisjoint(set(val["speaker"]))

imputer = SimpleImputer(strategy="median")
scaler = StandardScaler()

X_train = scaler.fit_transform(
    imputer.fit_transform(train[FEATURES])
)

# These are the same preprocessing steps as C2,
# without fitting a regression model.
correlation = np.corrcoef(X_train, rowvar=False)

corr_rows = []

for i in range(len(FEATURES)):
    for j in range(i + 1, len(FEATURES)):
        r = float(correlation[i, j])

        if np.isfinite(r):
            corr_rows.append({
                "feature_1": FEATURES[i],
                "feature_2": FEATURES[j],
                "pearson_r": r,
                "absolute_r": abs(r)
            })

corr_df = pd.DataFrame(corr_rows).sort_values(
    "absolute_r", ascending=False
).reset_index(drop=True)

# Eigenvalue-based condition number of the
# standardized design matrix, descriptive only.
singular_values = np.linalg.svd(
    X_train, compute_uv=False
)

positive_sv = singular_values[
    singular_values > 1e-10
]

condition_number = float(
    singular_values.max() / positive_sv.min()
)

rank = int(np.linalg.matrix_rank(X_train))

# Critical H2, standardized using fold 2 train statistics.
critical = val.loc[
    val["audio_id"].eq("030210164") &
    val["hypothesis_id"].eq("H2")
]

assert len(critical) == 1

z = scaler.transform(
    imputer.transform(critical[FEATURES])
)[0]

extremes = pd.DataFrame({
    "feature": FEATURES,
    "critical_H2_train_standardized_value": z,
    "absolute_standardized_value": np.abs(z)
}).sort_values(
    "absolute_standardized_value",
    ascending=False
).reset_index(drop=True)

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def save_csv(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing file differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

save_csv(coef_stability, STABILITY)
save_csv(corr_df, CORRELATIONS)
save_csv(extremes, EXTREMES)

report = {
    "status": "PASS",
    "analysis": "C2_coefficient_stability_multicollinearity",
    "fold_count": 5,
    "feature_count": 23,
    "sign_consistent_features": int(
        coef_stability["sign_consistent"].sum()
    ),
    "fold2_training_rows": 1600,
    "fold2_training_speakers": 16,
    "feature_matrix_rank": rank,
    "feature_matrix_condition_number": condition_number,
    "feature_pairs_abs_correlation_ge_0_9": int(
        (corr_df["absolute_r"] >= 0.9).sum()
    ),
    "critical_H2_max_abs_z": float(
        extremes["absolute_standardized_value"].max()
    ),
    "model_refitting": False,
    "coefficients_source_sha256": sha(COEFS),
    "contributions_source_sha256": sha(CONTRIB),
    "outputs_sha256": {
        path.name: sha(path)
        for path in [STABILITY, CORRELATIONS, EXTREMES]
    },
    "limitations": [
        "Coefficients describe conditional fitted model weights.",
        "Correlated predictors limit isolated feature interpretation.",
        "Only five training folds were examined.",
        "Fold 2 correlation analysis is diagnostic.",
        "No causal feature importance can be inferred."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_c2_coefficient_stability"] = "qc_pass"
progress["step6_c2_coefficient_stability_sha256"] = sha(REPORT)
progress["next_task"] = (
    "Review coefficient consistency, correlated predictors "
    "and extreme standardized feature values"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.7 COEFFICIENT STABILITY AUDIT ===")
print("Status: PASS")
print("Features:", len(FEATURES))
print(
    "Sign-consistent features:",
    report["sign_consistent_features"]
)
print("Fold 2 design rank:", rank)
print("Fold 2 condition number:", condition_number)
print(
    "Feature pairs |r| >= 0.9:",
    report["feature_pairs_abs_correlation_ge_0_9"]
)

print("\n=== TOP 15 COEFFICIENTS ===")
print(
    coef_stability.nlargest(
        15, "mean_absolute_coefficient"
    ).round(4).to_string(index=False)
)

print("\n=== TOP 15 FEATURE CORRELATIONS ===")
print(corr_df.head(15).round(4).to_string(index=False))

print("\n=== CRITICAL H2 EXTREME FEATURES ===")
print(extremes.head(12).round(4).to_string(index=False))

print("\nReport:", REPORT)
print("Checkpoint updated.")


In [ ]:

import json
import hashlib
from pathlib import Path
from datetime import datetime, timezone

import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

P = Path("/kaggle/working/acoustic_linguistic_research")
DATA = P / "data/modeling/modeling_dataset_0001-0400.csv"
SPLIT = P / "data/splits/speaker_groupkfold5_frozen.csv"
DESIGN = P / "configs/fluency_feature_design_v1.json"
C2 = P / "results/step05/c2_prompt_aware_fusion_ridge_oof_predictions.csv"
A = P / "results/step05/baseline_acoustic_oof_predictions.csv"
CHECK = P / "checkpoints/research_progress.json"

OUTDIR = P / "results/step06_final_tables"
DETAIL = OUTDIR / "step06_ood_diagnostic_per_hypothesis.csv"
SUMMARY = OUTDIR / "step06_ood_diagnostic_summary.csv"
REPORT = P / "reports/step06_ood_feature_audit_v1.json"

EXPECTED = {
    DATA: "7d20428a14d9e00b88f1a20b36b7147bf5b98234045a093172fef76b68276746",
    SPLIT: "5f87cdc1c2f69922bcb542ee9516f5f89168bf68402fbd2449b28fcf61a375e7",
    DESIGN: "626b0050f58f35fafa4f2efcd1888fda2a195454e544c4cb094cd50e796a1b83",
    C2: "2592e8d28c656db43280d6eea30d578cd488d75f0382e3ed083abf5e2ab86df9",
    A: "48407b4eac0f213e4ebf39d5b88aa131178e1aca3597cd41011cef168eac1595"
}

def sha(path):
    h = hashlib.sha256()
    with path.open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

for path, expected in EXPECTED.items():
    assert path.is_file()
    assert sha(path) == expected, f"SHA mismatch: {path}"

def read(path):
    return pd.read_csv(
        path,
        dtype={"audio_id": "string", "speaker": "string"}
    )

data = read(DATA)
split = read(SPLIT)
c2 = read(C2)
acoustic_pred = read(A)
design = json.loads(DESIGN.read_text(encoding="utf-8"))

IDS = ["pilot_index", "audio_id", "speaker"]
KEYS = IDS + ["hypothesis_id"]
FULL = KEYS + ["fluency", "validation_fold"]

LING = design["linguistic_feature_names"]
PROMPT = design["prompt_feature_names"]
ACOUSTIC = [
    c for c in data.columns
    if c not in KEYS + ["fluency"] + LING + PROMPT
]
FEATURES = ACOUSTIC + LING + PROMPT

assert len(FEATURES) == 23

df = data.merge(
    split[IDS + ["fluency", "validation_fold"]],
    on=IDS + ["fluency"],
    validate="many_to_one"
)

df = df.merge(
    c2[FULL + ["c2_prediction"]],
    on=FULL,
    validate="one_to_one"
)

df = df.merge(
    acoustic_pred[
        IDS + ["fluency", "validation_fold",
               "acoustic_ridge_prediction"]
    ],
    on=IDS + ["fluency", "validation_fold"],
    validate="many_to_one"
)

assert len(df) == 2000
assert df["audio_id"].nunique() == 400
assert df["speaker"].nunique() == 20

parts = []

for fold in range(1, 6):
    train = df.loc[df["validation_fold"] != fold]
    val = df.loc[df["validation_fold"] == fold].copy()

    assert len(train) == 1600
    assert len(val) == 400
    assert set(train["speaker"]).isdisjoint(
        set(val["speaker"])
    )

    imputer = SimpleImputer(strategy="median")
    scaler = StandardScaler()

    Xtrain = imputer.fit_transform(train[FEATURES])
    scaler.fit(Xtrain)

    Xval = imputer.transform(val[FEATURES])
    Z = scaler.transform(Xval)

    assert Z.shape == (400, 23)
    assert np.isfinite(Z).all()

    abs_z = np.abs(Z)
    ap = abs_z[:, :14]
    lp = abs_z[:, 14:]

    val["max_abs_z_all"] = abs_z.max(axis=1)
    val["max_abs_z_acoustic"] = ap.max(axis=1)
    val["max_abs_z_text_prompt"] = lp.max(axis=1)
    val["count_abs_z_gt5"] = (abs_z > 5).sum(axis=1)
    val["count_text_prompt_z_gt5"] = (lp > 5).sum(axis=1)
    val["any_text_prompt_z_gt5"] = (
        val["max_abs_z_text_prompt"] > 5
    )

    worst_idx = lp.argmax(axis=1)
    text_features = FEATURES[14:]
    val["largest_text_prompt_z_feature"] = [
        text_features[int(i)] for i in worst_idx
    ]

    parts.append(
        val[
            FULL + [
                "c2_prediction",
                "acoustic_ridge_prediction",
                "max_abs_z_all",
                "max_abs_z_acoustic",
                "max_abs_z_text_prompt",
                "count_abs_z_gt5",
                "count_text_prompt_z_gt5",
                "any_text_prompt_z_gt5",
                "largest_text_prompt_z_feature"
            ]
        ]
    )

detail = pd.concat(parts, ignore_index=True)
detail = detail.sort_values(
    ["pilot_index", "hypothesis_id"]
).reset_index(drop=True)

assert len(detail) == 2000
assert not detail.duplicated(KEYS).any()

detail["C2_absolute_error"] = (
    detail["c2_prediction"] - detail["fluency"]
).abs()
detail["A_absolute_error"] = (
    detail["acoustic_ridge_prediction"] -
    detail["fluency"]
).abs()

detail["C2_squared_error"] = (
    detail["c2_prediction"] - detail["fluency"]
) ** 2

detail["A_squared_error"] = (
    detail["acoustic_ridge_prediction"] -
    detail["fluency"]
) ** 2

# Primary values must reproduce existing fixed results.
assert np.isclose(
    detail["C2_absolute_error"].mean(),
    0.7782, atol=1e-4
)
assert np.isclose(
    detail["A_absolute_error"].mean(),
    0.9399, atol=1e-4
)

detail["diagnostic_group"] = np.where(
    detail["any_text_prompt_z_gt5"],
    "text_prompt_abs_z_gt5",
    "text_prompt_abs_z_le5"
)

summary_rows = []

for group, sub in detail.groupby("diagnostic_group"):
    summary_rows.append({
        "diagnostic_group": group,
        "hypotheses": int(len(sub)),
        "unique_audios": int(sub["audio_id"].nunique()),
        "unique_speakers": int(sub["speaker"].nunique()),
        "C2_MAE": float(sub["C2_absolute_error"].mean()),
        "C2_RMSE": float(
            np.sqrt(sub["C2_squared_error"].mean())
        ),
        "A_MAE": float(sub["A_absolute_error"].mean()),
        "A_RMSE": float(
            np.sqrt(sub["A_squared_error"].mean())
        ),
        "C2_minus_A_MAE": float(
            (sub["C2_absolute_error"] -
             sub["A_absolute_error"]).mean()
        ),
        "maximum_text_prompt_abs_z": float(
            sub["max_abs_z_text_prompt"].max()
        )
    })

summary = pd.DataFrame(summary_rows)

assert summary["hypotheses"].sum() == 2000

OUTDIR.mkdir(parents=True, exist_ok=True)
REPORT.parent.mkdir(parents=True, exist_ok=True)

def safe_save(frame, path):
    tmp = path.with_suffix(".tmp")
    frame.to_csv(
        tmp, index=False, encoding="utf-8",
        float_format="%.12g"
    )
    if path.exists():
        assert sha(path) == sha(tmp), (
            f"Existing output differs: {path}"
        )
        tmp.unlink()
    else:
        tmp.replace(path)

safe_save(detail, DETAIL)
safe_save(summary, SUMMARY)

critical = detail.loc[
    detail["audio_id"].eq("030210164") &
    detail["hypothesis_id"].eq("H2")
]

assert len(critical) == 1
assert critical["max_abs_z_text_prompt"].iloc[0] > 50

report = {
    "status": "PASS",
    "analysis": "training_fold_standardized_feature_extremes",
    "threshold_abs_z": 5.0,
    "threshold_role": "descriptive_diagnostic_not_formal_OOD",
    "total_hypotheses": 2000,
    "unique_audios": 400,
    "unique_speakers": 20,
    "hypotheses_above_threshold": int(
        detail["any_text_prompt_z_gt5"].sum()
    ),
    "audios_above_threshold": int(
        detail.loc[
            detail["any_text_prompt_z_gt5"],
            "audio_id"
        ].nunique()
    ),
    "model_refitting": False,
    "all_predictions_unchanged": True,
    "all_primary_observations_retained": True,
    "detail_sha256": sha(DETAIL),
    "summary_sha256": sha(SUMMARY),
    "source_sha256": {
        path.name: sha(path) for path in EXPECTED
    },
    "limitations": [
        "Z-score threshold of 5 is an exploratory diagnostic.",
        "All H1-H5 hypotheses of one audio are correlated.",
        "Feature extremeness and prediction errors are associated, not causally identified.",
        "Training distribution depends on each frozen fold.",
        "No additional model fitting or hyperparameter tuning."
    ]
}

if REPORT.exists():
    assert json.loads(
        REPORT.read_text(encoding="utf-8")
    ) == report
else:
    tmp = REPORT.with_suffix(".tmp")
    tmp.write_text(
        json.dumps(report, indent=2, ensure_ascii=False),
        encoding="utf-8"
    )
    tmp.replace(REPORT)

progress = json.loads(CHECK.read_text(encoding="utf-8"))
progress["step6_ood_feature_audit"] = "qc_pass"
progress["step6_ood_feature_audit_sha256"] = sha(SUMMARY)
progress["next_task"] = (
    "Assess feature-extreme prevalence and error association; "
    "complete Step 6 synthesis and checkpoint"
)
progress["updated_at"] = datetime.now(
    timezone.utc
).isoformat()

CHECK.write_text(
    json.dumps(progress, indent=2, ensure_ascii=False),
    encoding="utf-8"
)

print("=== STEP 6.8 FEATURE EXTREMENESS AUDIT ===")
print("Status: PASS")
print("Hypotheses:", len(detail))
print("Unique audios:", detail["audio_id"].nunique())
print("Models retrained: NO")
print("Primary observations excluded: NONE")

print("\n=== EXTREME FEATURE PREVALENCE ===")
print("Hypotheses with text/prompt |z| > 5:",
      report["hypotheses_above_threshold"])
print("Audios with text/prompt |z| > 5:",
      report["audios_above_threshold"])

print("\n=== ERROR BY FEATURE EXTREMENESS ===")
print(summary.round(4).to_string(index=False))

print("\n=== TOP 12 EXTREME HYPOTHESES ===")
print(
    detail.nlargest(
        12, "max_abs_z_text_prompt"
    )[
        [
            "audio_id", "speaker", "hypothesis_id",
            "max_abs_z_text_prompt",
            "largest_text_prompt_z_feature",
            "C2_absolute_error",
            "A_absolute_error"
        ]
    ].round(4).to_string(index=False)
)

print("\n=== CRITICAL H2 CHECK ===")
print(
    critical[
        [
            "max_abs_z_text_prompt",
            "C2_absolute_error",
            "A_absolute_error"
        ]
    ].round(4).to_string(index=False)
)

print("\nDetail CSV:", DETAIL)
print("Summary CSV:", SUMMARY)
print("Report:", REPORT)
print("Checkpoint updated.")
